# D3 Required PEFT/QLoRA Final Scope ? SLM Tuning, Demo Package, Repo Hygiene

**Use this notebook because PEFT/QLoRA tuning is included in this D3 submission.**

This notebook validates the PEFT/QLoRA evidence package:

1. `data/tuning/finetune_qa.jsonl` ? curated evidence-grounded tuning rows.
2. `reports/tables/d3_or_final_zero_shot_vs_tuned.csv` ? zero-shot baseline vs tuned-adapter status.
3. `reports/tables/d3_tuning_latency.csv` ? hardware/runtime feasibility evidence.

Important: this notebook does **not** claim a tuned adapter was trained unless the environment supports it. In the current local run, QLoRA training is recorded as **not feasible** because CUDA/PEFT/bitsandbytes/TRL are unavailable. That is intentionally honest evidence, not a fabricated tuned result.


In [1]:
# Common D3 setup.
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {"notebooks", "notebook"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORTS_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Reports/tables:", REPORTS_TABLES)

Project root: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent
Reports/tables: D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent\reports\tables


## 1. Required tuning/final scope

The original brief lists this as D4:

- PEFT/QLoRA on a small curated Q/A set.
- Tuning card: dataset size, epochs, learning rate, LoRA rank, hardware/time, license.
- Zero-shot vs tuned inside GraphRAG.
- Quantize/cache and final quality/latency table.
- 8-minute demo and 8-10 page final report.
- README one-command setup and pytest smoke tests.

This notebook is required final D3 integration evidence because PEFT/QLoRA tuning is included.

## 2. Tuning-data evidence

The tuning file is built from the D3 gold Q/A set and verified chunks:

- each row has an instruction, question/input, answer/output, source document, page range, evidence chunk id, and license note;
- answers are short synthesized target answers, not copied full-paper text;
- every row is tied back to a chunk id in `data/sample/sample_chunks.json`;
- the dataset is small, so it is suitable for format/adapter-pipeline demonstration, not for claiming a robust domain-tuned model.


In [2]:
# Validate finetune_qa.jsonl against required PEFT/QLoRA data fields.
# Works locally and on Kaggle if the exact dataset path is attached.
from IPython.display import display
import json
from pathlib import Path

KAGGLE_TUNING_PATH = Path('/kaggle/input/datasets/aayaehab/finetune-qa/finetune_qa.jsonl')
LOCAL_TUNING_PATH = PROJECT_ROOT / 'data' / 'tuning' / 'finetune_qa.jsonl'
CHUNKS_PATH = PROJECT_ROOT / 'data' / 'sample' / 'sample_chunks.json'

TUNING_PATH = KAGGLE_TUNING_PATH if KAGGLE_TUNING_PATH.exists() else LOCAL_TUNING_PATH

required_fields = {
    'instruction', 'input', 'output', 'source_document_id',
    'page_start', 'page_end', 'evidence_chunk_id', 'license_note'
}

assert TUNING_PATH.exists(), f'Missing required tuning file. Checked Kaggle={KAGGLE_TUNING_PATH} and local={LOCAL_TUNING_PATH}'

rows = [json.loads(line) for line in TUNING_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]

# Chunk-store validation is useful locally, but optional on Kaggle QLoRA because training only needs JSONL.
chunk_ids = None
if CHUNKS_PATH.exists():
    chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
    chunk_ids = {c.get('chunk_id') for c in chunks}
else:
    print('Chunk store not found; skipping chunk-id existence check. This is OK for Kaggle QLoRA training.')

validation_rows = []
for idx, row in enumerate(rows, start=1):
    missing = sorted(required_fields - set(row))
    chunk_ok = True if chunk_ids is None else row.get('evidence_chunk_id') in chunk_ids
    page_ok = str(row.get('page_start', '')).strip() != '' and str(row.get('page_end', '')).strip() != ''
    validation_rows.append({
        'row': idx,
        'evidence_chunk_id': row.get('evidence_chunk_id'),
        'source_document_id': row.get('source_document_id'),
        'missing_required_fields': ', '.join(missing),
        'chunk_exists': chunk_ok,
        'page_range_present': page_ok,
        'license_note_present': bool(str(row.get('license_note', '')).strip()),
    })

validation_df = pd.DataFrame(validation_rows)
print('Tuning path:', TUNING_PATH)
print(f'Tuning rows: {len(rows)}')
print('Rows with missing fields:', int((validation_df['missing_required_fields'] != '').sum()))
print('Rows with valid/unchecked evidence chunk id:', int(validation_df['chunk_exists'].sum()), '/', len(validation_df))
print('Rows with page range:', int(validation_df['page_range_present'].sum()), '/', len(validation_df))
assert len(rows) > 0, 'Tuning JSONL is empty.'
assert (validation_df['missing_required_fields'] == '').all(), 'Some tuning rows are missing required fields.'
assert validation_df['chunk_exists'].all(), 'Some tuning rows point to missing chunk IDs.'
assert validation_df['page_range_present'].all(), 'Some tuning rows have missing page ranges.'
display(validation_df.head(10))


Tuning rows: 15
Rows with missing fields: 0
Rows with valid evidence chunk id: 15 / 15
Rows with page range: 15 / 15


,row,evidence_chunk_id,source_document_id,missing_required_fields,chunk_exists,page_range_present,license_note_present
0,1,chunk_036782,treutlein_2026_real_world_energy_data_200_feed...,,True,True,True
1,2,chunk_006699,fatima_2020_fingerprints_climate_warming_cerea...,,True,True,True
2,3,chunk_036179,ravi_2025_citizen_centered_climate_intelligenc...,,True,True,True
3,4,chunk_023284,shah_2025_assessing_climate_vulnerability_risk...,,True,True,True
4,5,chunk_039454,scheller_2021_stakeholder_dynamics_residential...,,True,True,True
5,6,chunk_035742,smith_2019_current_future_role_haber_bosch_amm...,,True,True,True
6,7,chunk_021253,smith_2010_politics_social_ecological_resilien...,,True,True,True
7,8,chunk_006718,fatima_2020_fingerprints_climate_warming_cerea...,,True,True,True
8,9,chunk_033119,balcombe_2019_how_decarbonise_international_sh...,,True,True,True
9,10,chunk_036501,alkhayuon_2019_basin_bifurcations_oscillatory_...,,True,True,True


## 3. Tuning card / feasibility record

| Field | Current D3 evidence |
|---|---|
| Base model | Not trained locally; planned small instruct model for PEFT/QLoRA if GPU is available. |
| Dataset size | 15 evidence-grounded rows in `data/tuning/finetune_qa.jsonl`. |
| Epochs / LR / LoRA rank | Not executed locally, so not claimed. Must be filled only after a real GPU run. |
| Quantization | QLoRA intended, but `bitsandbytes` is unavailable in the current local `.venv`. |
| Hardware | Local run is CPU-only (`torch` CPU build, CUDA unavailable). |
| Runtime | Training runtime recorded as 0 / not-run in `d3_tuning_latency.csv`. |
| License constraints | Rows cite verified chunks/pages and use synthesized answers; no full-paper text copied. |
| Failure cases | Main blocker is environment feasibility, not model quality. |

Decision: keep the PEFT/QLoRA evidence in the repo, but report the tuned adapter as **not produced** unless the team reruns this on a CUDA machine with `peft`, `bitsandbytes`, and `trl` installed.


In [3]:
# Validate zero-shot vs tuned comparison and tuning latency/feasibility evidence.
# Works locally and on Kaggle. After Kaggle training, prefer /kaggle/working/outputs first.
from IPython.display import display
import importlib.util
from pathlib import Path

COMPARE_CANDIDATES = [
    Path('/kaggle/working/outputs/d3_or_final_zero_shot_vs_tuned.csv'),
    Path('/kaggle/working/reports/tables/d3_or_final_zero_shot_vs_tuned.csv'),
    Path('/kaggle/input/datasets/aayaehab/d3-or-final-zero-shot-vs-tuned/d3_or_final_zero_shot_vs_tuned.csv'),
    REPORTS_TABLES / 'd3_or_final_zero_shot_vs_tuned.csv',
]
LATENCY_CANDIDATES = [
    Path('/kaggle/working/outputs/d3_tuning_latency.csv'),
    Path('/kaggle/working/reports/tables/d3_tuning_latency.csv'),
    Path('/kaggle/input/datasets/aayaehab/d3-tuning-latency/d3_tuning_latency.csv'),
    REPORTS_TABLES / 'd3_tuning_latency.csv',
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

COMPARE_PATH = first_existing(COMPARE_CANDIDATES)
LATENCY_PATH = first_existing(LATENCY_CANDIDATES)

print('Comparison candidates checked:')
for p in COMPARE_CANDIDATES:
    print(' ', p, 'exists=', p.exists())
print('Latency candidates checked:')
for p in LATENCY_CANDIDATES:
    print(' ', p, 'exists=', p.exists())

assert COMPARE_PATH is not None, 'Missing comparison table in all known local/Kaggle paths.'
print('Using comparison table:', COMPARE_PATH)

compare_df = pd.read_csv(COMPARE_PATH)
required_compare_cols = {
    'method', 'training_status', 'dataset_rows', 'faithfulness',
    'answer_relevance', 'citation_correct', 'mean_latency_ms',
    'p95_latency_ms', 'model_or_adapter', 'notes'
}
assert required_compare_cols.issubset(compare_df.columns), 'Comparison CSV schema is incomplete.'
assert 'zero_shot_graphrag_baseline' in set(compare_df['method']), 'Missing zero-shot baseline row.'
assert 'qlora_tuned_adapter' in set(compare_df['method']), 'Missing QLoRA/tuned adapter status row.'

display_compare = compare_df.copy()
for col in ['faithfulness', 'answer_relevance', 'citation_correct', 'mean_latency_ms', 'p95_latency_ms']:
    display_compare[col] = display_compare[col].where(
        display_compare[col].notna(),
        'N/A - adapter not trained yet'
    )

print('=== Zero-shot vs PEFT/QLoRA status ===')
display(display_compare)

if LATENCY_PATH is not None:
    print('Using tuning latency table:', LATENCY_PATH)
    latency_df = pd.read_csv(LATENCY_PATH)
    display(latency_df)
else:
    latency_df = pd.DataFrame()
    print('No d3_tuning_latency.csv found yet. This is OK before the Kaggle training output cell runs.')

try:
    import torch
    cuda_available = bool(torch.cuda.is_available())
    torch_version = torch.__version__
except Exception:
    cuda_available = False
    torch_version = 'not_importable'

pkg_status = {
    'torch_version': torch_version,
    'cuda_available': cuda_available,
    'peft_installed': importlib.util.find_spec('peft') is not None,
    'bitsandbytes_installed': importlib.util.find_spec('bitsandbytes') is not None,
    'trl_installed': importlib.util.find_spec('trl') is not None,
}
print('Current environment check:', pkg_status)

qlora_row = compare_df[compare_df['method'] == 'qlora_tuned_adapter'].iloc[0]
if qlora_row['training_status'] != 'completed':
    print('Interpretation: PEFT/QLoRA data preparation exists, but this table does not yet contain a completed trained adapter.')
    print('Run D3_07_Kaggle_QLoRA_Tuning.ipynb through training/evaluation to create /kaggle/working/outputs/d3_or_final_zero_shot_vs_tuned.csv.')
else:
    print('Interpretation: QLoRA adapter row is completed and has real metric values.')


=== Zero-shot vs tuned status ===


,method,training_status,dataset_rows,faithfulness,answer_relevance,citation_correct,mean_latency_ms,p95_latency_ms,model_or_adapter,notes
0,zero_shot_graphrag_baseline,completed_baseline_only,15,0.69875,0.7275,0.75,269.45,378.12,"base RAG/GraphRAG pipeline, no fine-tuned adapter",Baseline summarized from reports/tables/d3_rag...
1,qlora_tuned_adapter,not_run_non_feasible_local_environment,15,NaN,NaN,NaN,NaN,NaN,not produced,Tuned comparison not fabricated. Non-feasibili...


=== Tuning feasibility / latency evidence ===


,stage,status,duration_sec,hardware,torch_version,cuda_available,peft_installed,bitsandbytes_installed,trl_installed,notes
0,environment_check,completed,0,not_available,2.12.0+cpu,False,False,False,False,Checked local project .venv capability for PEF...
1,tuning_dataset_build,completed,0,local CPU/file generation,2.12.0+cpu,False,False,False,False,Built 15 JSONL rows from D3 gold evidence.
2,qlora_training,not_run_non_feasible_local_environment,0,not_available,2.12.0+cpu,False,False,False,False,CUDA GPU not available; peft not installed; bi...


Current environment check: {'torch_version': '2.12.0+cpu', 'cuda_available': False, 'peft_installed': False, 'bitsandbytes_installed': False, 'trl_installed': False}
QLoRA adapter is NOT claimed as trained. This is correct for the current CPU-only environment.


## 4. Final demo and repo hygiene checklist

TODO verify before final submission/demo:

- README has one-command setup.
- `.env.example` exists and contains no secrets.
- Seeds/test data are included.
- pytest smoke tests pass.
- Demo questions are prepared: 2 graph wins, 1 retrieval-only, 1 safety case, 1 failure case.
- Final report includes architecture, experiments, ablations, failure cases, ethics/licensing, and future work.